In [10]:
#Code Setup
from pathlib import Path
from dotenv import load_dotenv
import os
import sys
import importlib

PROJECT_ROOT = Path(
    "/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()

load_dotenv(PROJECT_ROOT / ".env", override=True)

DB = os.getenv("NEO4J_DATABASE", "neo4j")

print("Project:", PROJECT_ROOT)
print("Neo4j configured:", bool(os.getenv("NEO4J_PASSWORD")))
print("Database:", DB)

Project: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Neo4j configured: True
Database: neo4j


In [11]:
#Vector RAG - Neo4j vector integration setup
%pip install -U \
    llama-index-vector-stores-neo4jvector \
    llama-index-embeddings-huggingface \
    sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [12]:
from llama_index.vector_stores.neo4jvector import Neo4jVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core import (
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
)

from llama_index.core.node_parser import SentenceSplitter

print("✅ Imports ready")

✅ Imports ready


In [4]:
#Loading the documents
DATA_FOLDERS = [
    PROJECT_ROOT / "data/people",
    PROJECT_ROOT / "data/projects",
    PROJECT_ROOT / "data/meetings",
    PROJECT_ROOT / "data/decisions",
    PROJECT_ROOT / "data/technical_docs",
]

all_files = []

for folder in DATA_FOLDERS:
    all_files.extend(
        sorted(folder.glob("*.md"))
    )

print("Corpus files:", len(all_files))

for folder in DATA_FOLDERS:
    print(
        f"{folder.name:15s}",
        len(list(folder.glob("*.md")))
    )

Corpus files: 33
people          10
projects        4
meetings        6
decisions       8
technical_docs  5


In [5]:
documents = SimpleDirectoryReader(
    input_files=[
        str(f)
        for f in all_files
    ]
).load_data()

print("Documents loaded:", len(documents))

Documents loaded: 33


In [6]:
#Configure Local Embeddings
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("✅ Embedding model ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5658.07it/s]


✅ Embedding model ready


In [ ]:
#Test Embedding dimension for the selected embedding model
test_embedding = embed_model.get_text_embedding(
    "embedding dimension test"
)

EMBED_DIM = len(test_embedding)

print("Embedding dimension:", EMBED_DIM)

Embedding dimension: 384


In [7]:
#Define Chunking Strategy
text_splitter = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=40,
)

nodes = text_splitter.get_nodes_from_documents(
    documents
)

print("Documents:", len(documents))
print("Chunks:", len(nodes))

#256 token chunks with 40 token overlap is a good balance between context and retrieval performance for our short documents

Documents: 33
Chunks: 33


In [8]:
#Inspect Chunks
for i, node in enumerate(nodes[:5]):

    print("=" * 90)
    print("Chunk:", i + 1)

    print(
        "File:",
        node.metadata.get("file_name")
    )

    print()
    print(node.text[:800])
    print()

Chunk: 1
File: person_alice.md

---
doc_id: doc_people_alice
doc_type: person_profile
entity_id: person_alice
owner: People Operations
---
# Alice Chen

**Title:** Principal Search Engineer  
**Team:** Search  
**Skills:** Kafka, Elasticsearch, Python  
**Projects:** Project Atlas, Project Phoenix

## Profile
Leads search ranking architecture and cross-team retrieval initiatives.

## Current responsibilities
Alice Chen contributes to Project Atlas, Project Phoenix. Their home team is Search.

Chunk: 2
File: person_bob.md

---
doc_id: doc_people_bob
doc_type: person_profile
entity_id: person_bob
owner: People Operations
---
# Bob Singh

**Title:** Senior ML Engineer  
**Team:** Recommendations  
**Skills:** Kubernetes, Python, Redis  
**Projects:** Project Phoenix, Project Atlas

## Profile
Builds low-latency recommendation serving and feature caching systems.

## Current responsibilities
Bob Singh contributes to Project Phoenix, Project Atlas. Their home team is Recommendations.

Chunk

In [ ]:
#Check Chunk Distribution per File
from collections import Counter

chunk_counts = Counter(
    node.metadata.get("file_name")
    for node in nodes
)

for file_name, count in sorted(
    chunk_counts.items()
):
    print(
        f"{file_name:45s}",
        count
    )

#every file becomes exactly one chunk, because our corpus is small

decision_d001.md                              1
decision_d002.md                              1
decision_d003.md                              1
decision_d004.md                              1
decision_d005.md                              1
decision_d006.md                              1
decision_d007.md                              1
decision_d008.md                              1
kubernetes_service_standard.md                1
meeting_m001.md                               1
meeting_m002.md                               1
meeting_m003.md                               1
meeting_m004.md                               1
meeting_m005.md                               1
meeting_m006.md                               1
person_alice.md                               1
person_bob.md                                 1
person_carol.md                               1
person_david.md                               1
person_elena.md                               1
person_farah.md                         

In [14]:
#Remove any old test vector rag experiment data
from graph.neo4j_client import get_driver

driver = get_driver()

with driver.session(database=DB) as session:

    count = session.run("""
        MATCH (n:VectorChunk)
        RETURN count(n) AS count
    """).single()["count"]

print("Existing VectorChunk nodes:", count)

driver.close()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: VectorChunk)} {position: line: 2, column: 18, offset: 18} for query: '\n        MATCH (n:VectorChunk)\n        RETURN count(n) AS count\n    '


Existing VectorChunk nodes: 0


In [15]:
#Clear the vector chunk nodes from the database
driver = get_driver()

with driver.session(database=DB) as session:

    session.run("""
        MATCH (n:VectorChunk)
        DETACH DELETE n
    """)

driver.close()

print("✅ Existing VectorChunk nodes cleared")

✅ Existing VectorChunk nodes cleared


In [16]:
#Dedicated index name
VECTOR_INDEX_NAME = "orgmind_vector_index"
VECTOR_NODE_LABEL = "VectorChunk"

In [17]:
#Inspect current indexes
driver = get_driver()

with driver.session(database=DB) as session:

    indexes = list(
        session.run("""
            SHOW INDEXES
            YIELD name, type, labelsOrTypes, properties
            RETURN name, type, labelsOrTypes, properties
            ORDER BY name
        """)
    )

for row in indexes:
    print(dict(row))

driver.close()

{'name': 'constraint_8f002f5b', 'type': 'RANGE', 'labelsOrTypes': ['__Entity__'], 'properties': ['id']}
{'name': 'constraint_d2d2cfa4', 'type': 'RANGE', 'labelsOrTypes': ['__Node__'], 'properties': ['id']}
{'name': 'entity', 'type': 'VECTOR', 'labelsOrTypes': ['__Entity__'], 'properties': ['embedding']}
{'name': 'index_1b9dcc97', 'type': 'LOOKUP', 'labelsOrTypes': None, 'properties': None}
{'name': 'index_460996c0', 'type': 'LOOKUP', 'labelsOrTypes': None, 'properties': None}


In [ ]:
# If orgmind_vector_index already exists from an earlier failed run, remove just that
driver = get_driver()

with driver.session(database=DB) as session:

    session.run("""
        DROP INDEX orgmind_vector_index IF EXISTS
    """)

driver.close()

print("✅ Old vector index removed if present")

In [18]:
#Configure neo4j as Vector Store
neo4j_vector_store = Neo4jVectorStore(
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    url=os.getenv("NEO4J_URI"),

    embedding_dimension=EMBED_DIM,

    database=DB,

    index_name=VECTOR_INDEX_NAME,

    node_label=VECTOR_NODE_LABEL,

    embedding_node_property="embedding",
    text_node_property="text",
)

print("✅ Neo4j Vector Store configured")

✅ Neo4j Vector Store configured


In [19]:
#Building Vector RAG Index
storage_context = StorageContext.from_defaults(
    vector_store=neo4j_vector_store
)

vector_index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

print("✅ Vector index built")

Generating embeddings: 100%|██████████| 33/33 [00:01<00:00, 20.39it/s]


✅ Vector index built


In [20]:
#Verify neo4j
driver = get_driver()

with driver.session(database=DB) as session:

    count = session.run("""
        MATCH (n:VectorChunk)
        RETURN count(n) AS count
    """).single()["count"]

print("VectorChunk nodes:", count)

driver.close()

VectorChunk nodes: 33


In [21]:
#Test
driver = get_driver()

with driver.session(database=DB) as session:

    rows = session.run("""
        MATCH (n:VectorChunk)
        RETURN
            n.text AS text,
            n.file_name AS file_name,
            size(n.embedding) AS embedding_dimension
        LIMIT 5
    """)

    for row in rows:
        print(dict(row))

driver.close()

{'text': '---\ndoc_id: doc_people_julia\ndoc_type: person_profile\nentity_id: person_julia\nowner: People Operations\n---\n# Julia Patel\n\n**Title:** Payments Tech Lead  \n**Team:** Payments  \n**Skills:** Airflow, Kafka, PostgreSQL  \n**Projects:** Project Mercury, Project Phoenix\n\n## Profile\nLeads payment-platform architecture and operational readiness.\n\n## Current responsibilities\nJulia Patel contributes to Project Mercury, Project Phoenix. Their home team is Payments.', 'file_name': 'person_julia.md', 'embedding_dimension': 384}
{'text': '---\ndoc_id: doc_project_atlas\ndoc_type: project_overview\nentity_id: project_atlas\nowner: Search\n---\n# Project Atlas\n\n**Status:** Production  \n**Owning team:** Search  \n**Project lead:** Alice Chen  \n**Core technologies:** Kafka, Elasticsearch, Python, Kubernetes\n\n## Purpose\nModernize product search ranking with fresh behavioral signals and a unified retrieval stack.\n\n## Contributors\nAlice Chen, Bob Singh, George Liu, Hannah

In [22]:
#Verifying neo4j vector index itself
driver = get_driver()

with driver.session(database=DB) as session:

    rows = session.run("""
        SHOW VECTOR INDEXES
        YIELD
            name,
            state,
            populationPercent,
            labelsOrTypes,
            properties

        WHERE name = $index_name

        RETURN
            name,
            state,
            populationPercent,
            labelsOrTypes,
            properties
    """,
    index_name=VECTOR_INDEX_NAME)

    for row in rows:
        print(dict(row))

driver.close()

{'name': 'orgmind_vector_index', 'state': 'ONLINE', 'populationPercent': 100.0, 'labelsOrTypes': ['VectorChunk'], 'properties': ['embedding']}


In [23]:
#Creating Vector RAG Retriever
vector_retriever = vector_index.as_retriever(
    similarity_top_k=5
)

print("✅ Vector retriever ready")

✅ Vector retriever ready


In [24]:
#Semantic test
question = "Why did Project Atlas adopt Kafka?"

results = vector_retriever.retrieve(
    question
)

for i, result in enumerate(
    results,
    start=1
):

    print("=" * 90)

    print(
        f"Rank {i} | "
        f"Score: {result.score:.4f}"
    )

    print(
        "File:",
        result.node.metadata.get(
            "file_name"
        )
    )

    print()

    print(result.node.text[:700])

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Rank 1 | Score: 0.9116
File: decision_d001.md

---
doc_id: doc_decision_d001
doc_type: architecture_decision
entity_id: decision_d001
project_id: project_atlas
date: 2026-02-12
---
# Adopt Kafka for Atlas change feed

**Decision ID:** decision_d001  
**Project:** Project Atlas  
**Technology:** Kafka  
**Proposed by:** Carol Martinez  
**Approved by:** Alice Chen  
**Date:** 2026-02-12

## Decision
Adopt Kafka for Atlas change feed.

## Rationale
Atlas needed replayable, ordered product and behavior updates. Kafka provided durable event history and decoupled producers from ranking consumers.

## Alternatives considered
Direct database polling, Managed queue without replay
Rank 2 | Score: 0.8825
File: streaming_platform_standard.md

---
doc_id: doc_tech_streaming_standard
doc_type: technical_document
owner: Data Platform
---
# Streaming Platform Standard

Acme AI standardizes high-volume event streaming on Kafka when replay, ordered partitions, and multiple independent consumers are req

In [25]:
#Graph Heavy test
question = (
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

results = vector_retriever.retrieve(
    question
)

for i, result in enumerate(
    results,
    start=1
):

    print("=" * 90)

    print(
        f"Rank {i} | "
        f"Score: {result.score:.4f}"
    )

    print(
        "File:",
        result.node.metadata.get(
            "file_name"
        )
    )

    print()

    print(result.node.text[:700])

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Rank 1 | Score: 0.8952
File: project_phoenix.md

---
doc_id: doc_project_phoenix
doc_type: project_overview
entity_id: project_phoenix
owner: Recommendations
---
# Project Phoenix

**Status:** Production  
**Owning team:** Recommendations  
**Project lead:** Farah Khan  
**Core technologies:** Kafka, Kubernetes, Redis, Python

## Purpose
Serve real-time personalized product recommendations using streaming behavior and low-latency feature access.

## Contributors
Alice Chen, Bob Singh, Carol Martinez, Farah Khan, Hannah Brooks, Julia Patel
Rank 2 | Score: 0.8920
File: kubernetes_service_standard.md

---
doc_id: doc_tech_kubernetes_standard
doc_type: technical_document
owner: Data Platform
---
# Kubernetes Service Deployment Standard

Kubernetes is the preferred deployment platform for continuously running services that need controlled rollouts, autoscaling, and standardized health checks.

Hannah Brooks owns the production-readiness checklist. Project Atlas runs ranking services on Kube

In [26]:
#Create reusable vector_search()
def vector_search(
    question: str,
    top_k: int = 5,
):

    retriever = vector_index.as_retriever(
        similarity_top_k=top_k
    )

    results = retriever.retrieve(
        question
    )

    return {
        "question": question,
        "top_k": top_k,
        "results": [
            {
                "rank": rank,
                "score": result.score,
                "file_name":
                    result.node.metadata.get(
                        "file_name"
                    ),
                "file_path":
                    result.node.metadata.get(
                        "file_path"
                    ),
                "text":
                    result.node.text,
            }
            for rank, result in enumerate(
                results,
                start=1
            )
        ],
    }

In [27]:
#Test
vector_search(
    "Why did Project Atlas adopt Kafka?"
)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


{'question': 'Why did Project Atlas adopt Kafka?',
 'top_k': 5,
 'results': [{'rank': 1,
   'score': 0.9115989208221436,
   'file_name': 'decision_d001.md',
   'file_path': '/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d001.md',
   'text': '---\ndoc_id: doc_decision_d001\ndoc_type: architecture_decision\nentity_id: decision_d001\nproject_id: project_atlas\ndate: 2026-02-12\n---\n# Adopt Kafka for Atlas change feed\n\n**Decision ID:** decision_d001  \n**Project:** Project Atlas  \n**Technology:** Kafka  \n**Proposed by:** Carol Martinez  \n**Approved by:** Alice Chen  \n**Date:** 2026-02-12\n\n## Decision\nAdopt Kafka for Atlas change feed.\n\n## Rationale\nAtlas needed replayable, ordered product and behavior updates. Kafka provided durable event history and decoupled producers from ranking consumers.\n\n## Alternatives considered\nDirect database polling, Managed queue without replay'},
  {'rank': 2,
   'score': 0.882

We're not using Neo4j graph relationships during the Vector-RAG baseline.

Even though everything now lives in Neo4j:

Neo4j
├── PERSON / PROJECT / SKILL / ...
│      ↑
│   Graph RAG
│
└── VectorChunk
       ↑
   Vector RAG

the baseline vector retriever only queries:

VectorChunk.embedding

through:

orgmind_vector_index

In [28]:
#Load 10 evaluation questions
import json

questions_path = (
    PROJECT_ROOT
    / "evaluation"
    / "questions.json"
)

evaluation_questions = json.loads(
    questions_path.read_text()
)

print("Questions:", len(evaluation_questions))

for q in evaluation_questions:
    print(
        f"Q{q['id']} [{q['category']}] "
        f"{q['question']}"
    )

Questions: 10
Q1 [single_fact] What database does Project Phoenix use for its online feature cache?
Q2 [single_fact] What is the purpose of Project Atlas?
Q3 [semantic] Why did Project Atlas adopt Kafka?
Q4 [relationship] Who worked on Project Phoenix and also has Kubernetes experience?
Q5 [relationship] Which people worked on both Project Atlas and Project Phoenix?
Q6 [multi_hop] Who approved the architecture decisions for technologies used by Project Atlas?
Q7 [relationship] Which projects were worked on by members of the Search team?
Q8 [multi_hop] Which technologies are used by projects led by Alice Chen?
Q9 [multi_hop] Who is the best person to consult about Kafka based on both project experience and documented technical decisions?
Q10 [multi_hop] Which architecture decisions affected projects owned by teams whose members also worked on Project Phoenix?


In [29]:
#Run Vector retrieval on all 10 questions

#Keep top_k=5 fixed for every question so the experiment stays consistent.
vector_retrieval_results = []

for item in evaluation_questions:

    result = vector_search(
        item["question"],
        top_k=5,
    )

    record = {
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "retrieval": result["results"],
    }

    vector_retrieval_results.append(record)

    print("=" * 100)
    print(
        f"Q{item['id']} "
        f"[{item['category']}]"
    )
    print(item["question"])

    print("\nTop-5 retrieved files:")

    for r in result["results"]:
        print(
            f"{r['rank']}. "
            f"{r['file_name']:40s} "
            f"score={r['score']:.4f}"
        )

    print()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vec

Q1 [single_fact]
What database does Project Phoenix use for its online feature cache?

Top-5 retrieved files:
1. decision_d004.md                         score=0.8992
2. recommendation_architecture.md           score=0.8810
3. project_phoenix.md                       score=0.8776
4. meeting_m004.md                          score=0.8652
5. decision_d005.md                         score=0.8522

Q2 [single_fact]
What is the purpose of Project Atlas?

Top-5 retrieved files:
1. project_atlas.md                         score=0.8805
2. search_architecture.md                   score=0.8673
3. meeting_m002.md                          score=0.8526
4. decision_d002.md                         score=0.8515
5. meeting_m001.md                          score=0.8448

Q3 [semantic]
Why did Project Atlas adopt Kafka?

Top-5 retrieved files:
1. decision_d001.md                         score=0.9116
2. streaming_platform_standard.md           score=0.8825
3. meeting_m001.md                          score=0.

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vec

Q4 [relationship]
Who worked on Project Phoenix and also has Kubernetes experience?

Top-5 retrieved files:
1. project_phoenix.md                       score=0.8952
2. kubernetes_service_standard.md           score=0.8920
3. recommendation_architecture.md           score=0.8698
4. project_ownership_and_contacts.md        score=0.8675
5. person_hannah.md                         score=0.8635

Q5 [relationship]
Which people worked on both Project Atlas and Project Phoenix?

Top-5 retrieved files:
1. project_ownership_and_contacts.md        score=0.8433
2. project_phoenix.md                       score=0.8432
3. project_atlas.md                         score=0.8359
4. streaming_platform_standard.md           score=0.8248
5. search_architecture.md                   score=0.8147

Q6 [multi_hop]
Who approved the architecture decisions for technologies used by Project Atlas?

Top-5 retrieved files:
1. meeting_m001.md                          score=0.8817
2. decision_d002.md                    

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vec

Q7 [relationship]
Which projects were worked on by members of the Search team?

Top-5 retrieved files:
1. project_atlas.md                         score=0.8667
2. person_george.md                         score=0.8576
3. search_architecture.md                   score=0.8570
4. person_alice.md                          score=0.8538
5. project_ownership_and_contacts.md        score=0.8379

Q8 [multi_hop]
Which technologies are used by projects led by Alice Chen?

Top-5 retrieved files:
1. person_alice.md                          score=0.8419
2. project_phoenix.md                       score=0.8332
3. project_atlas.md                         score=0.8249
4. search_architecture.md                   score=0.8198
5. project_ownership_and_contacts.md        score=0.8179



Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Q9 [multi_hop]
Who is the best person to consult about Kafka based on both project experience and documented technical decisions?

Top-5 retrieved files:
1. streaming_platform_standard.md           score=0.8511
2. decision_d001.md                         score=0.8510
3. decision_d005.md                         score=0.8487
4. recommendation_architecture.md           score=0.8438
5. meeting_m001.md                          score=0.8310

Q10 [multi_hop]
Which architecture decisions affected projects owned by teams whose members also worked on Project Phoenix?

Top-5 retrieved files:
1. project_ownership_and_contacts.md        score=0.8737
2. project_phoenix.md                       score=0.8621
3. meeting_m004.md                          score=0.8444
4. decision_d005.md                         score=0.8421
5. person_farah.md                          score=0.8369



In [30]:
#Save Vector retrieval results
vector_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "vector_retrieval_results.json"
)

vector_output_path.write_text(
    json.dumps(
        vector_retrieval_results,
        indent=2,
        default=str,
    )
)

print("Saved:", vector_output_path)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/vector_retrieval_results.json


In [31]:
#Compact retrieval summary
for item in vector_retrieval_results:

    print("=" * 100)

    print(
        f"Q{item['id']} "
        f"[{item['category']}]"
    )

    print(item["question"])

    print("\nRetrieved:")

    for r in item["retrieval"]:
        print(
            f"  {r['rank']}. "
            f"{r['file_name']} "
            f"({r['score']:.4f})"
        )

    print()

Q1 [single_fact]
What database does Project Phoenix use for its online feature cache?

Retrieved:
  1. decision_d004.md (0.8992)
  2. recommendation_architecture.md (0.8810)
  3. project_phoenix.md (0.8776)
  4. meeting_m004.md (0.8652)
  5. decision_d005.md (0.8522)

Q2 [single_fact]
What is the purpose of Project Atlas?

Retrieved:
  1. project_atlas.md (0.8805)
  2. search_architecture.md (0.8673)
  3. meeting_m002.md (0.8526)
  4. decision_d002.md (0.8515)
  5. meeting_m001.md (0.8448)

Q3 [semantic]
Why did Project Atlas adopt Kafka?

Retrieved:
  1. decision_d001.md (0.9116)
  2. streaming_platform_standard.md (0.8825)
  3. meeting_m001.md (0.8803)
  4. search_architecture.md (0.8756)
  5. project_atlas.md (0.8733)

Q4 [relationship]
Who worked on Project Phoenix and also has Kubernetes experience?

Retrieved:
  1. project_phoenix.md (0.8952)
  2. kubernetes_service_standard.md (0.8920)
  3. recommendation_architecture.md (0.8698)
  4. project_ownership_and_contacts.md (0.8675)
 

SETUP FOR CREATING AGENTIC RAG in Notebook 5

In [32]:
#Verify the existing Neo4j vector index
from graph.neo4j_client import get_driver
import os

DB = os.getenv("NEO4J_DATABASE", "neo4j")

driver = get_driver()

with driver.session(database=DB) as session:
    rows = session.run("""
        SHOW VECTOR INDEXES
        YIELD
            name,
            state,
            populationPercent,
            labelsOrTypes,
            properties

        WHERE name = "orgmind_vector_index"

        RETURN
            name,
            state,
            populationPercent,
            labelsOrTypes,
            properties
    """)

    for row in rows:
        print(dict(row))

driver.close()

{'name': 'orgmind_vector_index', 'state': 'ONLINE', 'populationPercent': 100.0, 'labelsOrTypes': ['VectorChunk'], 'properties': ['embedding']}


In [33]:
#Create retrieval/vector_retriever.py
from pathlib import Path

vector_retriever_code = r'''
"""
Reusable Vector RAG retriever for OrgMind.

This module connects to the EXISTING Neo4j vector index.
It does not ingest documents or rebuild embeddings.
"""

from functools import lru_cache
from typing import Any
import os

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.neo4jvector import Neo4jVectorStore

from config import (
    NEO4J_URI,
    NEO4J_USERNAME,
    NEO4J_PASSWORD,
)


DB = os.getenv(
    "NEO4J_DATABASE",
    "neo4j",
)

VECTOR_INDEX_NAME = "orgmind_vector_index"
VECTOR_NODE_LABEL = "VectorChunk"

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"


@lru_cache(maxsize=1)
def get_embed_model():
    """
    Load the local embedding model once per Python process.
    """

    return HuggingFaceEmbedding(
        model_name=EMBED_MODEL_NAME
    )


@lru_cache(maxsize=1)
def get_vector_index():
    """
    Connect LlamaIndex to the existing Neo4j vector index.

    No document ingestion happens here.
    """

    embed_model = get_embed_model()

    # Determine the embedding dimension from the model.
    embedding_dimension = len(
        embed_model.get_text_embedding(
            "embedding dimension probe"
        )
    )

    vector_store = Neo4jVectorStore(
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        url=NEO4J_URI,

        database=DB,

        index_name=VECTOR_INDEX_NAME,
        node_label=VECTOR_NODE_LABEL,

        embedding_dimension=embedding_dimension,

        embedding_node_property="embedding",
        text_node_property="text",
    )

    return VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        embed_model=embed_model,
    )


def retrieve_vector(
    question: str,
    top_k: int = 5,
) -> dict[str, Any]:
    """
    Retrieve the top-k semantically similar document chunks.

    Returns structured retrieval evidence without
    generating an LLM answer.
    """

    if top_k < 1:
        raise ValueError(
            "top_k must be greater than 0"
        )

    index = get_vector_index()

    retriever = index.as_retriever(
        similarity_top_k=top_k
    )

    retrieved_nodes = retriever.retrieve(
        question
    )

    results = []

    for rank, item in enumerate(
        retrieved_nodes,
        start=1,
    ):
        results.append({
            "rank": rank,
            "score": (
                float(item.score)
                if item.score is not None
                else None
            ),
            "file_name":
                item.node.metadata.get(
                    "file_name"
                ),
            "file_path":
                item.node.metadata.get(
                    "file_path"
                ),
            "doc_id":
                item.node.metadata.get(
                    "doc_id"
                ),
            "doc_type":
                item.node.metadata.get(
                    "doc_type"
                ),
            "text":
                item.node.text,
        })

    return {
        "question": question,
        "top_k": top_k,
        "result_count": len(results),
        "results": results,
    }
'''

output_path = (
    PROJECT_ROOT
    / "retrieval"
    / "vector_retriever.py"
)

output_path.write_text(
    vector_retriever_code
)

print("Saved:", output_path)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/retrieval/vector_retriever.py


In [34]:
#Import the module
import importlib
import retrieval.vector_retriever

importlib.reload(
    retrieval.vector_retriever
)

from retrieval.vector_retriever import (
    retrieve_vector,
    get_vector_index,
)

print(
    "✅ Vector retriever module imported"
)

✅ Vector retriever module imported


In [35]:
#Test an easy semantic question
result = retrieve_vector(
    "Why did Project Atlas adopt Kafka?",
    top_k=5,
)

print(
    "Results:",
    result["result_count"]
)

for row in result["results"]:

    print("=" * 90)

    print(
        f"Rank {row['rank']} | "
        f"Score: {row['score']:.4f}"
    )

    print(
        "File:",
        row["file_name"]
    )

    print()

    print(
        row["text"][:700]
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4726.26it/s]
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Results: 5
Rank 1 | Score: 0.9116
File: decision_d001.md

---
doc_id: doc_decision_d001
doc_type: architecture_decision
entity_id: decision_d001
project_id: project_atlas
date: 2026-02-12
---
# Adopt Kafka for Atlas change feed

**Decision ID:** decision_d001  
**Project:** Project Atlas  
**Technology:** Kafka  
**Proposed by:** Carol Martinez  
**Approved by:** Alice Chen  
**Date:** 2026-02-12

## Decision
Adopt Kafka for Atlas change feed.

## Rationale
Atlas needed replayable, ordered product and behavior updates. Kafka provided durable event history and decoupled producers from ranking consumers.

## Alternatives considered
Direct database polling, Managed queue without replay
Rank 2 | Score: 0.8825
File: streaming_platform_standard.md

---
doc_id: doc_tech_streaming_standard
doc_type: technical_document
owner: Data Platform
---
# Streaming Platform Standard

Acme AI standardizes high-volume event streaming on Kafka when replay, ordered partitions, and multiple independent consum

In [36]:
#Test graph-heavy question
result = retrieve_vector(
    (
        "Who worked on Project Phoenix "
        "and also has Kubernetes experience?"
    ),
    top_k=5,
)

for row in result["results"]:

    print(
        f"{row['rank']}. "
        f"{row['file_name']} "
        f"score={row['score']:.4f}"
    )

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


1. project_phoenix.md score=0.8952
2. kubernetes_service_standard.md score=0.8920
3. recommendation_architecture.md score=0.8698
4. project_ownership_and_contacts.md score=0.8675
5. person_hannah.md score=0.8635


In [37]:
#Compare the two modules
from retrieval.graph_retriever import (
    retrieve_graph,
)

from retrieval.vector_retriever import (
    retrieve_vector,
)

In [38]:
#Graph
graph_result = retrieve_graph(
    intent="person_project_skill",
    parameters={
        "project": "Project Phoenix",
        "skill": "Kubernetes",
    },
)

graph_result

{'intent': 'person_project_skill',
 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'},
 'result_count': 2,
 'results': [{'person': 'Bob Singh',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'},
  {'person': 'Hannah Brooks',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'}]}

In [39]:
#Vector
vector_result = retrieve_vector(
    (
        "Who worked on Project Phoenix "
        "and also has Kubernetes experience?"
    ),
    top_k=5,
)

vector_result

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


{'question': 'Who worked on Project Phoenix and also has Kubernetes experience?',
 'top_k': 5,
 'result_count': 5,
 'results': [{'rank': 1,
   'score': 0.8951500654220581,
   'file_name': 'project_phoenix.md',
   'file_path': '/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/projects/project_phoenix.md',
   'doc_id': None,
   'doc_type': None,
   'text': '---\ndoc_id: doc_project_phoenix\ndoc_type: project_overview\nentity_id: project_phoenix\nowner: Recommendations\n---\n# Project Phoenix\n\n**Status:** Production  \n**Owning team:** Recommendations  \n**Project lead:** Farah Khan  \n**Core technologies:** Kafka, Kubernetes, Redis, Python\n\n## Purpose\nServe real-time personalized product recommendations using streaming behavior and low-latency feature access.\n\n## Contributors\nAlice Chen, Bob Singh, Carol Martinez, Farah Khan, Hannah Brooks, Julia Patel'},
  {'rank': 2,
   'score': 0.8919565677642822,
   'file_name': 'kubernetes_service